<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Moyenne globale - RMSE public : 4.8225
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bibliothèques
</div>

In [1]:
import pandas as pd
import numpy as np

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des documents de données temporelles et de test
</div>

In [2]:
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet") # Données temporelles (Index 1 : 'station', Index 2 : 'commit_at', Colonnes : 'bikes' et 'stands')
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet") # A compléter (Index 1 : 'station', Index 2 : 'commit_at', Colonnes : 'bikes_+15min' à 'bikes_+120min')

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Création d'un dataset de test pour évaluer sa performance
</div>
On récupère les 8 dernières valeurs

In [3]:
rang_depuis_la_fin = history_toulouse_train.groupby(level="station").cumcount(ascending=False)
history_toulouse_apprentissage = history_toulouse_train[rang_depuis_la_fin >= 8].copy()

On créé l'ensemble de validation avec la dernière valeur connue + les 8 à prédire

In [4]:
last_9 = history_toulouse_train.groupby(level="station").tail(9).copy()

On remet l'index sous forme de colonnes

In [5]:
last_9 = last_9.reset_index()

On calule la somme cummulée

In [6]:
last_9["horizon"] = last_9.groupby("station").cumcount()

On crée une ligne par station et on récupère les 8 valeurs futures

In [7]:
history_toulouse_validation = (
    last_9
    .pivot(
        index="station",
        columns="horizon",
        values="bikes"
    )
    .drop(columns=0)
    .rename(columns={
        1: "bikes_+15min",
        2: "bikes_+30min",
        3: "bikes_+45min",
        4: "bikes_+60min",
        5: "bikes_+75min",
        6: "bikes_+90min",
        7: "bikes_+105min",
        8: "bikes_+120min"
    })
)

Récupérer la date de la dernière mesure de l'ensemble d'apprentissage par station

In [8]:
cutoff_times = last_9[last_9["horizon"] == 0].set_index("station")["commit_at"]

Ajouter commit_at

In [9]:
history_toulouse_validation["commit_at"] = cutoff_times

Mettre station + commit_at en MultiIndex

In [10]:
history_toulouse_validation = history_toulouse_validation.reset_index().set_index(["station", "commit_at"]).sort_index()

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Prédictions
</div>
Arrondi de la moyenne du nombre de vélos par station

In [11]:
moyenne_par_station = history_toulouse_apprentissage.groupby(level="station")["bikes"].mean().round().astype(int)

On extrait le nom de chaque station dans test

In [12]:
station_index = history_toulouse_test.index.get_level_values("station")

On attribue chaque station de test à sa moyenne calculée précédemment (ca évite les erreures si les stations de test ne sont pas dans le même ordre que dans train)

In [13]:
valeurs_moyennes = station_index.map(moyenne_par_station)

On remplit toutes les lignes avec une seule valeur

In [14]:
colonnes_cibles = history_toulouse_test.columns
for col in colonnes_cibles:
    history_toulouse_test[col] = valeurs_moyennes

Calcul du score RMSE

In [15]:
rmse = np.sqrt(np.nanmean((history_toulouse_test.values - history_toulouse_validation.values)**2))
print(f'RMSE',rmse)

RMSE 5.218755987081368


<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
On recalcule tout avec TOUTES les données d'entraînement fournies
</div>

In [16]:
moyenne_par_station = history_toulouse_train.groupby(level="station")["bikes"].mean().round().astype(int)
station_index = history_toulouse_test.index.get_level_values("station")
valeurs_moyennes = station_index.map(moyenne_par_station)
colonnes_cibles = history_toulouse_test.columns
for col in colonnes_cibles:
    history_toulouse_test[col] = valeurs_moyennes

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [17]:
history_toulouse_test.to_parquet("BA_prediction_1_naive.parquet", index=True)
history_toulouse_test

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,6,6,6,6,6,6,6,6
00002 - LAFAYETTE,2026-07-31 23:45:00,10,10,10,10,10,10,10,10
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,4,4,4,4,4,4,4,4
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,6,6,6,6,6,6,6,6
00003 - POMME,2025-01-05 05:45:00,10,10,10,10,10,10,10,10
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0
